In [ ]:
from utils import read_fit, read_xrr

In [ ]:
graded_model = read_fit("graded/graded_fit.pkl", material="znpc")

In [ ]:
graded_model.plot()

In [ ]:
from utils.models import (
    AnisotropyObjective,
    CurveFitter,
    ReflectModel,
    Transform,
    configure_refloxide_fitting,
)
from utils.refloxide_profile import model_bookended_refloxide

configure_refloxide_fitting(use_rust=True, parallel=False)

In [ ]:
mat = graded_model.model.structure[1]

mat.tau_vac.setp(bounds=(0, 30))

In [ ]:
import matplotlib.pyplot as plt

fitter = CurveFitter(graded_model)
fitter.fit("differential_evolution", workers=8, x0 = [v.value for v in graded_model.varying_parameters()])
fitter.objective.plot()
plt.show()
fitter.objective.model.structure.plot()
plt.show()

In [ ]:
print(fitter.objective.varying_parameters())

In [ ]:
objective = fitter.objective
objective.model.structure.components[1].density_bulk.setp(vary=False, bounds=(1, 2.5))
objective.model.structure.components[1].density_si.constraint = None
objective.model.structure.components[1].density_si.setp(vary=True, bounds=(0.5, 2.5))
objective.model.structure.components[1].density_vac.constraint = None
objective.model.structure.components[1].density_vac.setp(vary=True, bounds=(0.5, 2.5))

print(objective.varying_parameters())

In [ ]:
fitter = CurveFitter(objective)

fitter.fit(
    "differential_evolution",
    workers=8,
    x0=[v.value for v in objective.varying_parameters()],
)
fitter.objective.plot()
plt.show()
fitter.objective.model.structure.plot()
plt.show()

In [ ]:
import pickle

from utils import models_root

with open(
    models_root / "xrr" / "znpc" / "graded" / "graded_fit.pkl",
    "wb",
) as f:
    pickle.dump(fitter.objective, f)